# Diabetic Readmission: Missingness, Leakage, and Model Selection

PhDAI 733 · Deliverable 2 · Sai Teja Bandaru and Rashi Jain

This notebook is a **runner**. All logic lives in `src/` and all parameters in `config.yaml`,
so nothing is defined twice and a change in one place takes effect everywhere.

| module | responsibility |
|---|---|
| `src/data_loader.py` | download, parse, cohort construction, feature engineering |
| `src/preprocessing.py` | the three missingness regimes as leakage-safe pipelines |
| `src/evaluation.py` | diagnostics, cross-validation, tuning, subgroup audit |
| `src/visualization.py` | figures |

Runtime is roughly 10–15 minutes on a Colab CPU instance.

## 1. Setup

In [ ]:
# Colab opens this notebook without the rest of the repository, so fetch it.
# Always sync to the latest main: a stale _repo from an earlier run in the same
# session would otherwise shadow any fix that has since been merged.
import os, sys, subprocess
from pathlib import Path

REPO   = "https://github.com/rashijain/diabetic-readmission-audit.git"
BRANCH = "main"

if not Path("src").exists():
    if Path("_repo/.git").exists():
        subprocess.run(["git", "-C", "_repo", "fetch", "--quiet", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", "_repo", "reset", "--hard", "--quiet",
                        f"origin/{BRANCH}"], check=True)
        print("existing clone updated to latest", BRANCH)
    else:
        subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO, "_repo"],
                       check=True)
        print("repository cloned")
    os.chdir("_repo")

sys.path.insert(0, os.getcwd())

# Drop any src modules cached from an earlier run, so the freshly pulled code is used.
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader   import CFG, load_raw, build_cohort
from src.preprocessing import build_preprocessor, REGIMES
from src.evaluation    import (missingness_diagnostics, run_cv, tune,
                               subgroup_audit, expected_calibration_error, get_models)
from src.visualization import (plot_missingness, plot_leakage_and_regimes,
                               plot_subgroups, plot_decision_curve, net_benefit)

sns.set_theme(style="whitegrid", context="notebook")
SEED = CFG["project"]["random_state"]
Path("outputs").mkdir(exist_ok=True)
print(f"{CFG['project']['name']} | seed {SEED} | regimes {REGIMES}")
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

## 2. Data and cohort

`build_cohort` applies the exclusions and feature engineering defined in `config.yaml`:
expired and hospice discharges removed, zero-variance medication columns dropped, ICD-9
codes collapsed into nine clinical categories, and the medication-burden features derived.

It also resets the index, which later subgroup indexing depends on.

In [ ]:
df = build_cohort(load_raw())
df.head()

## 3. Missingness diagnostics

Tests whether missingness is Missing Not At Random by comparing the readmission rate
where each field is recorded against where it is absent.

In [ ]:
mnar = missingness_diagnostics(df)
display(mnar)
print("\nweight has p = 1.000 - it is the NULL CONTROL, not an MNAR feature.")
print("Its null result is what shows the other four reflect clinical signal.")

In [ ]:
plot_missingness(mnar)

## 4. Leakage ablation (RQ1)

46% of encounters belong to patients appearing more than once, so a naive random split
puts the same patient in train and test. Identical pipelines are run under encounter-level
and patient-grouped cross-validation.

In [ ]:
res_leaky,   _        = run_cv(df, "explicit_mnar", "leaky_encounter")
res_grouped, oof_expl = run_cv(df, "explicit_mnar", "grouped_patient")

leakage = pd.concat([res_leaky, res_grouped], ignore_index=True)
auc = leakage.pivot(index="Model", columns="Split", values="ROC_AUC")
inflation = (auc["leaky_encounter"] - auc["grouped_patient"]).to_frame("ROC_AUC_inflation")
display(inflation.round(4))
print("fold-to-fold SD:", leakage["ROC_SD"].round(4).tolist())
print("\nInflation is far below the noise floor - the leakage hypothesis does not hold here.")

## 5. Missingness regimes (RQ2)

Three preprocessing regimes under patient-grouped validation: drop the high-missingness
features, impute them modally, or encode absence explicitly.

In [ ]:
res_drop,   oof_drop   = run_cv(df, "drop_mnar",       "grouped_patient")
res_impute, oof_impute = run_cv(df, "standard_impute", "grouped_patient")

results = pd.concat([leakage, res_drop, res_impute], ignore_index=True)
results.to_csv("outputs/all_results.csv", index=False)

OOF = {"drop_mnar": oof_drop, "standard_impute": oof_impute, "explicit_mnar": oof_expl}
import pickle
pickle.dump(OOF, open("outputs/oof.pkl", "wb"))

grouped = results[results["Split"] == "grouped_patient"]
display(grouped.pivot(index="Regime", columns="Model", values="PR_AUC")
               .reindex(list(REGIMES)).round(4))

In [ ]:
plot_leakage_and_regimes(results)

## 6. Hyperparameter tuning

Grid search over the selected model under patient-grouped CV. The grid, the split count
and the scoring metric all come from `config.yaml`. The objective is average precision,
not accuracy: the positive class is 11.39% of the cohort, so predicting "no readmission"
throughout would score 88.61% accuracy while identifying nobody.

In [ ]:
grid_search, baseline = tune(df)

## 7. Subgroup audit (RQ3)

In [ ]:
subgroups = subgroup_audit(df, OOF)
by_race = subgroups[(subgroups.Attribute == "race_grp") &
                    (subgroups.Regime == "explicit_mnar")]
display(by_race[["Subgroup", "N", "Prevalence", "ROC_AUC", "ECE", "FNR_at_threshold"]].round(4))

In [ ]:
plot_subgroups(subgroups)

## 8. Clinical utility

Decision curve analysis: net benefit across plausible intervention thresholds, with false
positives penalised at the odds each threshold implies.

In [ ]:
plot_decision_curve(df["target"].values, OOF)

threshold = CFG["validation"]["clinical_decision_threshold"]
for regime in REGIMES:
    nb_value = net_benefit(df["target"].values, OOF[regime]["HistGBDT"],
                           np.array([threshold]))[0]
    print(f"net benefit @ {threshold:.0%}, {regime:16s}: {nb_value:+.5f}")
prevalence = df["target"].mean()
print(f"net benefit @ {threshold:.0%}, {'intervene on all':16s}: "
      f"{prevalence - (1 - prevalence) * (threshold / (1 - threshold)):+.5f}")

## Ethical considerations

- **Potential biases.** `race` is as recorded by hospital staff, not self-identified, so the
  subgroup labels are themselves a product of the documentation process under study.
- **Subgroup performance.** Patients with no recorded race show the lowest discrimination and
  a false negative rate near 0.80 against roughly 0.52 for the best-served group. The model is
  weakest where documentation is thinnest, and that is not randomly distributed.
- **Calibration.** Expected calibration error for Asian and Hispanic patients is roughly ten
  times that for Caucasian patients, so one threshold does not carry one meaning across groups.
- **Interpretability.** Gradient boosting is not interpretable by inspection; a clinician acting
  on a flag is owed a reason. Logistic regression is retained as a transparent reference.
- **Deployment constraint.** Additive only. The model may raise patients onto an intervention
  list; it must never remove a patient a clinician has flagged.
- **Data.** The raw dataset is read only. Exclusions and derived features are applied to an
  in-memory copy; nothing is written back to `diabetic_data.csv`, and it is not committed.